# LAB 05 — Advanced Data Quality & Cross-System Validation
### Retail Customer 360 | Databricks Solutions Architect Interview Lab

**Language:** English throughout. **Runtime:** Databricks Free Edition, Serverless, PySpark  
**Input:** Five `retail_customer360.bronze.*_raw` Delta tables from LAB 04, plus `silver.customers_current` from LAB 03  
**Output:** Five validated Silver tables, five quarantine tables, five superseded-record tables, lineage/quality metrics, and analytics-ready reconciliation views.  
**Scope:** Full-snapshot data quality. **Do not ingest LAB 06 CDC files or modify `silver.customers_scd2`.**

> **Business challenge:** A European retailer combines Salesforce CRM, SAP ERP, e-commerce, payment-gateway, and marketing activity. Business users want reliable Customer 360 and Sales Analytics. Raw source data includes orphan references, invalid amounts, inconsistent formats, late arrivals, and duplicate business keys. We must **preserve evidence**, **publish trustworthy Silver datasets**, and **quantify what has been rejected**.

**Run sequentially.** No external network access is needed: LAB 04 already ingested the public GitHub CSVs into Delta Bronze.

## Learning outcomes and architecture

You will practice explicit data-quality contracts, distributed PySpark expressions, safe `try_cast` parsing, SQL three-valued logic, deterministic window-based deduplication, left-join foreign-key validation, quarantine patterns, end-to-end record reconciliation, and **business-oriented exception analysis**.

```text
LAB 02: bronze.customers_raw → LAB 03: silver.customers_scd2 → silver.customers_current
                                                                 │
LAB 04: products_raw ───────────► Row validation ──► Dedup ──► products_clean
LAB 04: orders_raw ─────────────► Row validation ──► Dedup ──► Customer FK ──► orders_clean
LAB 04: order_items_raw ────────► Row validation ──► Dedup ──► Order/Product FK ──► order_items_clean
LAB 04: payments_raw ───────────► Row validation ──► Dedup ──► Order/Customer FK ──► payments_clean
LAB 04: customer_interactions_raw► Row validation ──► Dedup ──► Customer FK ──► customer_interactions_clean
                                         │                  │
                                    quarantine         superseded
                                         └──── Metrics / Reconciliation / Audit ────┘
```

**Important distinction:** An invalid record is quarantined; a valid but superseded duplicate is recorded separately. Both remain auditable and are counted against the Bronze source. An amount mismatch across systems is **not automatically fraud or bad data**: we publish exception reports for investigation.

**The historical dimension from LAB 03 remains untouched.** The current customer view is used only as a valid-key reference.

## STEP 01 — Configure Unity Catalog, source contracts, and safe rerun behavior

**Goal:** Fail early if any prerequisite is missing. All names are fully qualified; Spark SQL will use UTC when interpreting ISO-8601 timestamps.  

**Rerun policy:** By default, already-created LAB 05 Silver tables are not overwritten. The first run creates them. To deliberately rebuild this **snapshot lab**, set `ALLOW_REBUILD_LAB05_TABLES=True` after confirming that LAB 06 has **not** modified the tables. Never use this switch casually after incremental/CDC exercises.

In [ ]:
# STEP 01 — Set fixed catalog/schema names and validate prerequisites.
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from functools import reduce

CATALOG = "retail_customer360"
BRONZE = f"{CATALOG}.bronze"
SILVER = f"{CATALOG}.silver"
CUSTOMERS_CURRENT = f"{SILVER}.customers_current"

# These are LAB 04 initial-snapshot files, not future CDC feeds.
SOURCE_CONTRACTS = {
    "products": ("products_raw", 500, "product_id"),
    "orders": ("orders_raw", 15000, "order_id"),
    "order_items": ("order_items_raw", 40000, "order_item_id"),
    "payments": ("payments_raw", 15500, "payment_id"),
    "customer_interactions": ("customer_interactions_raw", 8000, "interaction_id"),
}

ALLOW_REBUILD_LAB05_TABLES = False
LAB_RUN_ID = "LAB05_INITIAL_SNAPSHOT_V1"
VALID_COUNTRIES = ["ES", "PT", "FR", "DE"]
MONEY_TOLERANCE = F.lit("0.01").cast("decimal(18,2)")

spark.conf.set("spark.sql.session.timeZone", "UTC")
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql("USE SCHEMA silver")

for domain, (table, expected, key) in SOURCE_CONTRACTS.items():
    fqn = f"{BRONZE}.{table}"
    assert spark.catalog.tableExists(fqn), f"Complete LAB 04 first: {fqn} is missing"
    actual = spark.table(fqn).count()
    assert actual == expected, (
        f"Unexpected {fqn} row count: {actual}, expected {expected}. "
        "LAB 05 assumes the original LAB 04 snapshots."
    )

assert spark.catalog.tableExists(CUSTOMERS_CURRENT), (
    f"Complete LAB 03 first: {CUSTOMERS_CURRENT} does not exist."
)
assert spark.table(CUSTOMERS_CURRENT).groupBy("customer_id").count().filter("count > 1").count() == 0
print("Prerequisites verified: 79,000 source rows across five Bronze tables.")
print("Valid customer reference:", CUSTOMERS_CURRENT)
print("Silver overwrite allowed:", ALLOW_REBUILD_LAB05_TABLES)


## STEP 02 — Verify exact source column contracts

**Goal:** Prevent hidden schema drift. The LAB 04 files reside under `data/crm/` in GitHub, although they represent five different systems. They have already been loaded into the corresponding Bronze Delta tables.  

We keep original source attributes as strings until the Silver conversion stage; metadata columns added during ingestion are retained for lineage.

In [ ]:
# STEP 02 — Match real CSV headers and preserve LAB 04 provenance.
SOURCE_COLUMNS = {
    "products": ["product_id", "sku", "product_name", "category", "subcategory", "brand",
                 "unit_price_eur", "cost_price_eur", "supplier_id", "active_flag",
                 "created_at", "updated_at", "source_system", "notes"],
    "orders": ["order_id", "customer_id", "order_timestamp", "order_status", "sales_channel",
               "store_id", "country", "currency", "subtotal_eur", "discount_eur",
               "shipping_eur", "total_amount_eur", "updated_at", "source_system", "notes"],
    "order_items": ["order_item_id", "order_id", "product_id", "quantity", "unit_price_eur",
                    "gross_amount_eur", "line_discount_eur", "line_total_eur", "currency",
                    "created_at", "updated_at", "source_system", "notes"],
    "payments": ["payment_id", "order_id", "customer_id", "payment_timestamp", "payment_type",
                 "payment_status", "payment_method", "amount_eur", "currency",
                 "gateway_reference", "updated_at", "source_system", "notes"],
    "customer_interactions": ["interaction_id", "customer_id", "interaction_timestamp",
                              "ingestion_timestamp", "channel", "event_type", "campaign_id",
                              "interaction_outcome", "device_type", "country", "source_system", "notes"],
}

# LAB 04 adds these fields; retain them when building Silver datasets.
SOURCE_METADATA = [
    "source_file", "source_file_url", "source_repository", "source_domain",
    "source_file_sha256", "ingestion_batch_id", "bronze_ingestion_timestamp",
    "ingestion_layer",
]

for domain, source_fields in SOURCE_COLUMNS.items():
    tbl = f"{BRONZE}.{SOURCE_CONTRACTS[domain][0]}"
    actual_cols = spark.table(tbl).columns
    assert actual_cols == source_fields + SOURCE_METADATA, (
        f"Column contract mismatch for {domain}: {actual_cols}"
    )
    print(f"{domain}: {len(source_fields)} business columns + {len(SOURCE_METADATA)} lineage columns")


## STEP 03 — Define reusable quality, parsing, and deduplication utilities

**Goal:** Keep the implementation consistent across five domains without creating five unrelated, difficult-to-maintain notebooks.

**Design principles:**

1. `raw_payload_json` records the original Bronze source fields before normalization. Bronze itself is never changed.
2. Empty strings become `NULL`; selected code fields are standardized to uppercase.
3. `try_cast` parses values without aborting the entire batch when a malformed field is encountered.
4. Quality rules produce **all** applicable error codes, not just the first.
5. Deduplication happens **only after row validation**, using a documented business-key and tie-breaking policy.
6. The selected winner is not necessarily the objectively correct business version when timestamps are ambiguous; source event sequencing is addressed in LAB 06.

In [ ]:
# STEP 03A — Generic cleansing and safe type-conversion utilities.

def bronze_to_working(domain):
    """Read an unchanged Bronze table and attach an auditable raw representation."""
    source = spark.table(f"{BRONZE}.{SOURCE_CONTRACTS[domain][0]}")
    business_cols = SOURCE_COLUMNS[domain]
    return (source
        .withColumn("raw_payload_json", F.to_json(
            F.struct(*[F.col(c) for c in business_cols]),
            options={"ignoreNullFields": "false"}
        ))
        .withColumn("source_record_hash", F.sha2(F.col("raw_payload_json"), 256)))


def normalize_fields(df, uppercase=(), lowercase=()):
    """Trim source strings, map whitespace-only fields to NULL, normalize selected enums."""
    for colname, dtype in df.dtypes:
        # Source ingestion timestamps added by LAB 04 are not modified.
        if dtype == "string" and colname not in (
            "raw_payload_json", "source_record_hash",
            "source_file", "source_file_url", "source_repository", "source_domain",
            "source_file_sha256", "ingestion_batch_id", "ingestion_layer"
        ):
            clean = F.trim(F.col(colname))
            df = df.withColumn(colname, F.when(clean == "", F.lit(None)).otherwise(clean))
    for colname in uppercase:
        df = df.withColumn(colname, F.upper(F.col(colname)))
    for colname in lowercase:
        df = df.withColumn(colname, F.lower(F.col(colname)))
    return df


def safely_parse(df, decimals=(), timestamps=(), integers=(), booleans=()):
    """Create _parsed columns; retain raw source text for diagnostic comparisons."""
    types = ({c: "DECIMAL(18,2)" for c in decimals}
             | {c: "TIMESTAMP" for c in timestamps}
             | {c: "INT" for c in integers}
             | {c: "BOOLEAN" for c in booleans})
    for colname, sql_type in types.items():
        df = df.withColumn(f"{colname}_parsed", F.expr(
            f"try_cast(`{colname}` AS {sql_type})"
        ))
    return df


def invalid_required_text(column):
    """Source text is null or empty after trimming."""
    return F.col(column).isNull()


def invalid_required_parse(column):
    """A required string was missing or was unparseable."""
    return F.col(column).isNull() | F.col(f"{column}_parsed").isNull()


def flag(rule_code, violation):
    """Turn a boolean violation into a nullable error-code expression."""
    return F.when(F.coalesce(violation, F.lit(False)), F.lit(rule_code))


def apply_quality_rules(df, rules):
    """Keep all quality failures in an array; mark an actionable status."""
    return (df
        .withColumn("dq_errors", F.array_compact(F.array(
            *[flag(rule_code, violation) for rule_code, violation in rules]
        )))
        .withColumn("dq_status", F.when(
            F.size(F.col("dq_errors")) == 0, F.lit("VALID")
        ).otherwise(F.lit("INVALID"))))


def quality_result_counts(df):
    """Return counts of rows that pass or fail all local rules."""
    return {
        "valid": df.filter(F.col("dq_status") == "VALID").count(),
        "invalid": df.filter(F.col("dq_status") == "INVALID").count(),
    }

print("Reusable cleaning, parsing, and rule helpers are ready.")


In [ ]:
# STEP 03B — Deterministic ranking and foreign-key quality utilities.

def deduplicate_valid(df, business_key):
    """Rank valid rows by the most recent available source timestamp.

    Different source domains expose different event/update date fields. A stable
    source hash is used as a reproducible technical tie-breaker. If two rows have
    identical source attributes, they are interchangeable for this lab.
    """
    date_cols = [f"{c}_parsed" for c in (
        "updated_at", "created_at", "payment_timestamp", "interaction_timestamp",
        "order_timestamp", "ingestion_timestamp"
    ) if f"{c}_parsed" in df.columns]
    ordering = ([F.col(c).desc_nulls_last() for c in date_cols]
                + [F.col("source_record_hash").desc()])
    window = Window.partitionBy(business_key).orderBy(*ordering)
    ranked = df.withColumn("duplicate_rank", F.row_number().over(window))
    winner = ranked.filter(F.col("duplicate_rank") == 1).drop("duplicate_rank")
    superseded = (ranked.filter(F.col("duplicate_rank") > 1)
        .withColumn("dq_status", F.lit("SUPERSEDED"))
        .drop("duplicate_rank"))
    return winner, superseded


def add_fk_violation(df, column, reference_df, reference_key, error_code):
    """Attach an FK violation, using distinct reference keys to avoid row multiplication."""
    ref = reference_df.select(
        F.col(reference_key).alias("__known_fk")
    ).filter(F.col("__known_fk").isNotNull()).distinct()
    joined = df.join(F.broadcast(ref), F.col(column) == F.col("__known_fk"), "left")
    bad_fk = F.col("__known_fk").isNull()
    return (joined
        .withColumn("dq_errors", F.when(
            bad_fk, F.concat(F.col("dq_errors"), F.array(F.lit(error_code)))
        ).otherwise(F.col("dq_errors")))
        .withColumn("dq_status", F.when(
            F.size(F.col("dq_errors")) == 0, "VALID"
        ).otherwise("INVALID"))
        .drop("__known_fk"))


def split_fk_result(df):
    """Separate FK-valid records and rejected orphans (without silent drops)."""
    return (df.filter(F.col("dq_status") == "VALID"),
            df.filter(F.col("dq_status") == "INVALID"))


def union_quarantine(local_bad, fk_bad):
    """Union local-rule failures and FK failures with a compatible column contract."""
    return local_bad.unionByName(fk_bad, allowMissingColumns=True)

print("Deduplication and referential-integrity helpers are ready.")


## STEP 04 — Build typed working DataFrames for each source

**Goal:** Standardize each source independently, preserving original raw values in `raw_payload_json`.  

**Important:** Original source columns remain strings until the final Silver projection. We add parsed counterparts such as `amount_eur_parsed`, so invalid source values can be inspected in quarantine. All monetary fields are parsed as **DECIMAL(18,2)**, not FLOAT or DOUBLE, to avoid binary floating-point surprises when reconciling currency.

In [ ]:
# STEP 04 — Define the typed working datasets.
working = {}

working["products"] = safely_parse(
    normalize_fields(bronze_to_working("products"), lowercase=["active_flag"]),
    decimals=["unit_price_eur", "cost_price_eur"],
    timestamps=["created_at", "updated_at"],
    booleans=["active_flag"]
)
working["orders"] = safely_parse(
    normalize_fields(bronze_to_working("orders"),
                     uppercase=["order_status", "sales_channel", "country", "currency"]),
    decimals=["subtotal_eur", "discount_eur", "shipping_eur", "total_amount_eur"],
    timestamps=["order_timestamp", "updated_at"]
)
working["order_items"] = safely_parse(
    normalize_fields(bronze_to_working("order_items"), uppercase=["currency"]),
    decimals=["unit_price_eur", "gross_amount_eur", "line_discount_eur", "line_total_eur"],
    timestamps=["created_at", "updated_at"],
    integers=["quantity"]
)
working["payments"] = safely_parse(
    normalize_fields(bronze_to_working("payments"),
                     uppercase=["payment_type", "payment_status", "payment_method", "currency"]),
    decimals=["amount_eur"],
    timestamps=["payment_timestamp", "updated_at"]
)
working["customer_interactions"] = safely_parse(
    normalize_fields(bronze_to_working("customer_interactions"),
                     uppercase=["channel", "event_type", "interaction_outcome", "country"]),
    timestamps=["interaction_timestamp", "ingestion_timestamp"]
)

for domain, df in working.items():
    print(f"{domain}: {df.count():,} source records")

# The source ingestion_timestamp belongs to Marketing; LAB 04 separately added
# bronze_ingestion_timestamp to all five sources.
assert "ingestion_timestamp_parsed" in working["customer_interactions"].columns
assert "bronze_ingestion_timestamp" in working["customer_interactions"].columns


## STEP 05 — Declare local quality rules for all five domains

**Goal:** Reject only demonstrably invalid *row-level* values. Cross-source FK rules are applied later, in dependency order.

**Rule choices are explicit LAB assumptions, not universal industry standards:**

| Domain | Examples of blocking rules | Non-blocking observations |
|---|---|---|
| Products | Missing product ID/SKU/category, non-positive prices, malformed timestamps | Product markup/discount business review |
| Orders | Invalid ID, timestamp, status, currency, amount, arithmetic | Country may differ from customer's address (shipping/travel) |
| Order items | Unknown IDs, nonpositive quantity, malformed price, incorrect line arithmetic | A valid order may have some quarantined items |
| Payments | Unknown IDs, invalid type/status, incorrect sign for charge/refund, malformed timestamps | Multiple attempts and refunds per order are legitimate |
| Interactions | Invalid customer ID, timestamp, channel or event type | Late arrival is an observability signal, **not automatically invalid** |

**Do not assume one payment per order.** Amounts must be reconciled with status, charge/refund type, and business policy. Invalid/unknown foreign keys are checked after reference tables have been constructed.

In [ ]:
# STEP 05A — Validate SAP product attributes.
p = working["products"]
product_rules = [
    ("PRD001_MISSING_ID", invalid_required_text("product_id")),
    ("PRD002_MISSING_SKU", invalid_required_text("sku")),
    ("PRD003_MISSING_NAME", invalid_required_text("product_name")),
    ("PRD004_MISSING_CATEGORY", invalid_required_text("category")),
    ("PRD005_INVALID_UNIT_PRICE", invalid_required_parse("unit_price_eur") |
     (F.col("unit_price_eur_parsed") <= 0)),
    ("PRD006_INVALID_COST", invalid_required_parse("cost_price_eur") |
     (F.col("cost_price_eur_parsed") < 0)),
    ("PRD007_INVALID_ACTIVE_FLAG", invalid_required_parse("active_flag")),
    ("PRD008_INVALID_CREATED_AT", invalid_required_parse("created_at")),
    ("PRD009_INVALID_UPDATED_AT", invalid_required_parse("updated_at")),
]
validated_products = apply_quality_rules(p, product_rules)
display(validated_products.groupBy("dq_status").count())


In [ ]:
# STEP 05B — Validate e-commerce orders and monetary identity.
o = working["orders"]
formula_total = (F.col("subtotal_eur_parsed") - F.col("discount_eur_parsed")
                 + F.col("shipping_eur_parsed"))
order_rules = [
    ("ORD001_MISSING_ID", invalid_required_text("order_id")),
    ("ORD002_MISSING_CUSTOMER", invalid_required_text("customer_id")),
    ("ORD003_INVALID_TIMESTAMP", invalid_required_parse("order_timestamp")),
    ("ORD004_INVALID_UPDATED_AT", invalid_required_parse("updated_at")),
    ("ORD005_INVALID_STATUS", F.col("order_status").isNull() |
     ~F.col("order_status").isin("COMPLETED", "SHIPPED", "PENDING", "CANCELLED", "RETURNED")),
    ("ORD006_INVALID_COUNTRY", F.col("country").isNull() |
     ~F.col("country").isin(*VALID_COUNTRIES)),
    ("ORD007_INVALID_CURRENCY", F.col("currency").isNull() | (F.col("currency") != "EUR")),
    ("ORD008_INVALID_SUBTOTAL", invalid_required_parse("subtotal_eur") |
     (F.col("subtotal_eur_parsed") < 0)),
    ("ORD009_INVALID_DISCOUNT", invalid_required_parse("discount_eur") |
     (F.col("discount_eur_parsed") < 0)),
    ("ORD010_INVALID_SHIPPING", invalid_required_parse("shipping_eur") |
     (F.col("shipping_eur_parsed") < 0)),
    ("ORD011_INVALID_TOTAL", invalid_required_parse("total_amount_eur") |
     (F.col("total_amount_eur_parsed") < 0)),
    ("ORD012_TOTAL_FORMULA_MISMATCH", F.col("subtotal_eur_parsed").isNotNull() &
     F.col("discount_eur_parsed").isNotNull() &
     F.col("shipping_eur_parsed").isNotNull() &
     F.col("total_amount_eur_parsed").isNotNull() &
     (F.abs(formula_total - F.col("total_amount_eur_parsed")) > MONEY_TOLERANCE)),
]
validated_orders = apply_quality_rules(o, order_rules)
display(validated_orders.groupBy("dq_status").count())


In [ ]:
# STEP 05C — Validate order items, quantity and line-level arithmetic.
i = working["order_items"]
expected_gross = F.col("quantity_parsed") * F.col("unit_price_eur_parsed")
expected_net_line = F.col("gross_amount_eur_parsed") - F.col("line_discount_eur_parsed")
item_rules = [
    ("ITM001_MISSING_ID", invalid_required_text("order_item_id")),
    ("ITM002_MISSING_ORDER", invalid_required_text("order_id")),
    ("ITM003_MISSING_PRODUCT", invalid_required_text("product_id")),
    ("ITM004_INVALID_QTY", invalid_required_parse("quantity") | (F.col("quantity_parsed") <= 0)),
    ("ITM005_INVALID_UNIT_PRICE", invalid_required_parse("unit_price_eur") |
     (F.col("unit_price_eur_parsed") <= 0)),
    ("ITM006_INVALID_GROSS", invalid_required_parse("gross_amount_eur") |
     (F.col("gross_amount_eur_parsed") < 0)),
    ("ITM007_INVALID_DISCOUNT", invalid_required_parse("line_discount_eur") |
     (F.col("line_discount_eur_parsed") < 0)),
    ("ITM008_INVALID_LINE_TOTAL", invalid_required_parse("line_total_eur") |
     (F.col("line_total_eur_parsed") < 0)),
    ("ITM009_INVALID_CURRENCY", F.col("currency").isNull() | (F.col("currency") != "EUR")),
    ("ITM010_INVALID_CREATED_AT", invalid_required_parse("created_at")),
    ("ITM011_INVALID_UPDATED_AT", invalid_required_parse("updated_at")),
    ("ITM012_GROSS_MISMATCH", F.col("quantity_parsed").isNotNull() &
     F.col("unit_price_eur_parsed").isNotNull() &
     F.col("gross_amount_eur_parsed").isNotNull() &
     (F.abs(expected_gross - F.col("gross_amount_eur_parsed")) > MONEY_TOLERANCE)),
    ("ITM013_NET_MISMATCH", F.col("gross_amount_eur_parsed").isNotNull() &
     F.col("line_discount_eur_parsed").isNotNull() &
     F.col("line_total_eur_parsed").isNotNull() &
     (F.abs(expected_net_line - F.col("line_total_eur_parsed")) > MONEY_TOLERANCE)),
]
validated_items = apply_quality_rules(i, item_rules)
display(validated_items.groupBy("dq_status").count())


In [ ]:
# STEP 05D — Validate payment gateway states and monetary signs.
pay = working["payments"]
known_status = (F.col("payment_status").isin("CAPTURED", "PENDING", "DECLINED", "SUCCEEDED"))
charge = F.col("payment_type") == "CHARGE"
refund = F.col("payment_type") == "REFUND"
payment_rules = [
    ("PAY001_MISSING_ID", invalid_required_text("payment_id")),
    ("PAY002_MISSING_ORDER", invalid_required_text("order_id")),
    ("PAY003_MISSING_CUSTOMER", invalid_required_text("customer_id")),
    ("PAY004_INVALID_TIMESTAMP", invalid_required_parse("payment_timestamp")),
    ("PAY005_INVALID_UPDATED_AT", invalid_required_parse("updated_at")),
    ("PAY006_INVALID_TYPE", F.col("payment_type").isNull() | ~(charge | refund)),
    ("PAY007_INVALID_STATUS", F.col("payment_status").isNull() | ~known_status),
    ("PAY008_INVALID_CURRENCY", F.col("currency").isNull() | (F.col("currency") != "EUR")),
    ("PAY009_INVALID_AMOUNT", invalid_required_parse("amount_eur") |
     (charge & (F.col("amount_eur_parsed") <= 0)) |
     (refund & (F.col("amount_eur_parsed") >= 0))),
    ("PAY010_INVALID_STATUS_FOR_TYPE", (refund & (F.col("payment_status") != "SUCCEEDED")) |
     (charge & (F.col("payment_status") == "SUCCEEDED"))),
]
validated_payments = apply_quality_rules(pay, payment_rules)
display(validated_payments.groupBy("dq_status").count())


In [ ]:
# STEP 05E — Validate marketing interaction format and event taxonomy.
m = working["customer_interactions"]
valid_events = [
    "SMS_RECEIVED", "APP_SESSION", "SMS_CLICK", "SUPPORT_CALL", "PUSH_OPEN",
    "COMPLAINT", "EMAIL_CLICK", "CART_ABANDONED", "EMAIL_OPEN",
    "PAGE_VIEW", "PRODUCT_VIEW", "UNSUBSCRIBE"
]
interaction_rules = [
    ("MKT001_MISSING_ID", invalid_required_text("interaction_id")),
    ("MKT002_MISSING_CUSTOMER", invalid_required_text("customer_id")),
    ("MKT003_INVALID_EVENT_TIMESTAMP", invalid_required_parse("interaction_timestamp")),
    ("MKT004_INVALID_INGESTION_TIMESTAMP", invalid_required_parse("ingestion_timestamp")),
    ("MKT005_INVALID_CHANNEL", F.col("channel").isNull() |
     ~F.col("channel").isin("SMS", "EMAIL", "WEB", "APP", "PHONE")),
    ("MKT006_INVALID_EVENT_TYPE", F.col("event_type").isNull() |
     ~F.col("event_type").isin(*valid_events)),
    ("MKT007_INVALID_COUNTRY", F.col("country").isNull() |
     ~F.col("country").isin(*VALID_COUNTRIES)),
]
validated_interactions = apply_quality_rules(m, interaction_rules)
display(validated_interactions.groupBy("dq_status").count())


## STEP 06 — Profile local-rule outcomes and inspect actual rejected records

**Goal:** See measured counts before we join sources. The error-code distribution is a multiset: one row can fail several rules, so the sum of violations may exceed the number of quarantined records. These measurements are computed from the Bronze rows, not manually hard-coded from the source manifest.

In [ ]:
# STEP 06 — Report rule failures and a small sample from each source.
validated = {
    "products": validated_products,
    "orders": validated_orders,
    "order_items": validated_items,
    "payments": validated_payments,
    "customer_interactions": validated_interactions,
}
local_quality_counts = {}
for domain, df in validated.items():
    counts = quality_result_counts(df)
    local_quality_counts[domain] = counts
    print(f"{domain:23s} VALID={counts['valid']:,} | INVALID={counts['invalid']:,}")

    print("Top validation codes for", domain)
    display(df.select(F.explode("dq_errors").alias("dq_code"))
        .groupBy("dq_code").count().orderBy(F.desc("count")).limit(12))

# Inspect concrete examples of row-level failures without modifying their source.
display(validated_products.filter("dq_status='INVALID'")
        .select("product_id", "category", "unit_price_eur", "dq_errors").limit(15))
display(validated_orders.filter("dq_status='INVALID'")
        .select("order_id", "customer_id", "order_status", "total_amount_eur", "dq_errors").limit(15))


## STEP 07 — Deduplicate the **row-valid** candidate records

**Goal:** Choose one row per primary/business key for a *current* Silver snapshot.  

Only valid rows are considered for ranking, and the non-selected rows are explicitly retained in `_superseded`. A later LAB will demonstrate CDC ordering; the timestamp/hash tie-breaker here is a defensible deterministic **snapshot convention**, not a production-grade CDC sequence.

In [ ]:
# STEP 07 — Split local quarantine, deduplicate valid rows, and retain losers.
local_bad = {}
selected = {}
superseded = {}
for domain, df in validated.items():
    business_key = SOURCE_CONTRACTS[domain][2]
    local_bad[domain] = df.filter(F.col("dq_status") == "INVALID")
    valid_candidates = df.filter(F.col("dq_status") == "VALID")
    selected[domain], superseded[domain] = deduplicate_valid(valid_candidates, business_key)
    print(f"{domain}: {valid_candidates.count():,} valid rows → "
          f"{selected[domain].count():,} selected; "
          f"{superseded[domain].count():,} superseded")
    assert selected[domain].groupBy(business_key).count().filter("count > 1").count() == 0

print("No source rows were discarded silently.")


## STEP 08 — Build authoritative foreign-key reference sets

**Goal:** Use the **validated current customer** representation from LAB 03. This catches both unknown customer IDs and IDs that exist in Bronze but were quarantined during the CRM cleansing lab.

FK reference sets are built from the progressively accepted Silver candidates in dependency order. An order is accepted only if its customer is valid. An item is accepted only if both its order and product are accepted. A payment must reference an accepted order and the same customer assigned to that order. Marketing interactions must reference an accepted customer.

**Nuance:** A mismatch between a customer's residential country and the country of a purchase is reported later as an *anomaly*, not automatically rejected (international delivery/travel may be legitimate).

In [ ]:
# STEP 08 — Read the current customer dimension as a governed reference.
customer_ref = (spark.table(CUSTOMERS_CURRENT)
    .select("customer_id", "country")
    .filter(F.col("customer_id").isNotNull())
    .dropDuplicates(["customer_id"]))

assert customer_ref.count() > 0
print("Available valid customer IDs:", customer_ref.count())

# Products have no foreign-key dependencies in this lab.
accepted = {}
fk_bad = {}
accepted["products"] = selected["products"]
fk_bad["products"] = selected["products"].limit(0)

# Orders depend on valid Customer IDs, not on raw CRM keys.
orders_checked = add_fk_violation(
    selected["orders"], "customer_id", customer_ref, "customer_id",
    "FK_ORD_CUSTOMER_NOT_IN_SILVER"
)
accepted["orders"], fk_bad["orders"] = split_fk_result(orders_checked)
print("Orders accepted:", accepted["orders"].count(),
      "Order FK exceptions:", fk_bad["orders"].count())


## STEP 09 — Validate order-item references against accepted orders and products

**Goal:** Detect orphans that only become visible **after** upstream quality controls. This intentionally introduces *cascading quarantine*: a line item may reference a genuine order ID, but if its order fails a critical rule, the line cannot yet be published as trusted Silver data.

In [ ]:
# STEP 09 — Order items need both parent order and product.
items_checked = add_fk_violation(
    selected["order_items"], "order_id",
    accepted["orders"], "order_id", "FK_ITM_ORDER_NOT_IN_SILVER"
)
items_checked = add_fk_violation(
    items_checked, "product_id",
    accepted["products"], "product_id", "FK_ITM_PRODUCT_NOT_IN_SILVER"
)
accepted["order_items"], fk_bad["order_items"] = split_fk_result(items_checked)
print("Order items accepted:", accepted["order_items"].count(),
      "Order-item FK exceptions:", fk_bad["order_items"].count())
display(fk_bad["order_items"].select(
    "order_item_id", "order_id", "product_id", "dq_errors"
).limit(25))


## STEP 10 — Validate payment references and ownership

**Goal:** Payments can have **multiple attempts per order**. Therefore, we validate the existence of the parent order and whether `payment.customer_id` matches the accepted order's `customer_id`; we **do not** require payment totals to equal order totals as a blocking row-level rule.

Refunds are negative, captured charges are positive, and payment status controls what counts as settled. Monetary reconciliation comes later as a **diagnostic** rather than an uncontrolled cascade of quarantines.

In [ ]:
# STEP 10 — Validate order existence, then the payment/order customer relationship.
payments_checked = add_fk_violation(
    selected["payments"], "order_id",
    accepted["orders"], "order_id", "FK_PAY_ORDER_NOT_IN_SILVER"
)

# Use an explicit alias to avoid collisions between source.customer_id and order.customer_id.
accepted_order_customers = (accepted["orders"]
    .select(F.col("order_id").alias("__order_id_ref"),
            F.col("customer_id").alias("__order_customer_ref")))
payments_checked = (payments_checked
    .join(F.broadcast(accepted_order_customers),
          F.col("order_id") == F.col("__order_id_ref"), "left")
    .withColumn("dq_errors", F.when(
        F.col("__order_id_ref").isNotNull()
        & (F.col("customer_id") != F.col("__order_customer_ref")),
        F.concat("dq_errors", F.array(F.lit("FK_PAY_CUSTOMER_MISMATCH")))
    ).otherwise(F.col("dq_errors")))
    .withColumn("dq_status", F.when(F.size("dq_errors") == 0, "VALID").otherwise("INVALID"))
    .drop("__order_id_ref", "__order_customer_ref"))

accepted["payments"], fk_bad["payments"] = split_fk_result(payments_checked)
print("Payments accepted:", accepted["payments"].count(),
      "Payment FK exceptions:", fk_bad["payments"].count())


## STEP 11 — Validate marketing interactions against accepted customers

**Goal:** Keep engagement history only when it can be attributed to a valid CRM customer. We intentionally **do not** quarantine valid late-arriving events. Latency is analyzed separately in STEP 19.

In [ ]:
# STEP 11 — Validate Marketing → CRM foreign keys.
interactions_checked = add_fk_violation(
    selected["customer_interactions"], "customer_id",
    customer_ref, "customer_id", "FK_MKT_CUSTOMER_NOT_IN_SILVER"
)
accepted["customer_interactions"], fk_bad["customer_interactions"] = split_fk_result(
    interactions_checked
)
print("Interactions accepted:", accepted["customer_interactions"].count(),
      "Interaction FK exceptions:", fk_bad["customer_interactions"].count())


## STEP 12 — Reconcile every Bronze source record into exactly one outcome

**Invariant for each source:**  

`bronze_count = clean_count + local_quarantine + FK_quarantine + superseded_count`

**Why this matters:** A pipeline that just calls `.filter()` and writes the survivors can silently lose records. Our accounting distinguishes business-valid records from rejected records and non-selected duplicates. We verify the invariant **before** writing any Silver tables.

In [ ]:
# STEP 12 — Demonstrate row-level end-to-end reconciliation before publication.
quarantine = {}
reconciliation_rows = []
for domain, (raw_table, expected, key) in SOURCE_CONTRACTS.items():
    quarantine[domain] = union_quarantine(local_bad[domain], fk_bad[domain])
    c_clean = accepted[domain].count()
    c_local = local_bad[domain].count()
    c_fk = fk_bad[domain].count()
    c_superseded = superseded[domain].count()
    total_accounted = c_clean + c_local + c_fk + c_superseded
    assert total_accounted == expected, (
        f"Record loss or multiplication detected in {domain}: "
        f"expected={expected}, accounted={total_accounted}"
    )
    reconciliation_rows.append({
        "domain": domain,
        "bronze_rows": expected,
        "clean_rows": c_clean,
        "local_quarantine_rows": c_local,
        "fk_quarantine_rows": c_fk,
        "total_quarantine_rows": c_local + c_fk,
        "superseded_rows": c_superseded,
        "accounted_rows": total_accounted,
    })

df_reconciliation = spark.createDataFrame(reconciliation_rows)
display(df_reconciliation.orderBy("domain"))
assert sum(r["bronze_rows"] for r in reconciliation_rows) == 79000
print("All 79,000 LAB 04 source rows have exactly one documented outcome.")


## STEP 13 — Publish typed, business-ready Silver schemas

**Goal:** Convert validated string columns into usable `DECIMAL`, `TIMESTAMP`, `INT`, and `BOOLEAN` columns. We keep key lineage fields (`source_file`, source file hash, batch ID, source-row fingerprint), plus an `silver_processed_timestamp`.

**Governance:** Quarantine retains the source string values, parsed values, original JSON payload, and violation codes; **Bronze continues to be the immutable evidence source**. Superseded rows remain in their own Delta tables.

**Snapshot policy:** `products_clean`, `orders_clean`, `order_items_clean`, `payments_clean`, and `customer_interactions_clean` represent a *quality-controlled initial snapshot*, not the CDC-enabled current state after LAB 06.

In [ ]:
# STEP 13A — Project normalized and typed attributes using the original business names.
PARSED_FIELDS = {
    "products": ["unit_price_eur", "cost_price_eur", "active_flag", "created_at", "updated_at"],
    "orders": ["subtotal_eur", "discount_eur", "shipping_eur", "total_amount_eur", "order_timestamp", "updated_at"],
    "order_items": ["quantity", "unit_price_eur", "gross_amount_eur", "line_discount_eur", "line_total_eur", "created_at", "updated_at"],
    "payments": ["amount_eur", "payment_timestamp", "updated_at"],
    "customer_interactions": ["interaction_timestamp", "ingestion_timestamp"],
}


def project_silver_clean(df, domain):
    """Replace raw string representations with validated parsed Spark types."""
    typed = set(PARSED_FIELDS[domain])
    fields = [
        F.col(f"{c}_parsed").alias(c) if c in typed else F.col(c)
        for c in SOURCE_COLUMNS[domain]
    ]
    return df.select(
        *fields,
        *[F.col(c) for c in SOURCE_METADATA],
        "source_record_hash",
        F.lit("VALID").alias("dq_status"),
        F.lit(LAB_RUN_ID).alias("silver_batch_id"),
        F.current_timestamp().alias("silver_processed_timestamp")
    )

silver_clean = {domain: project_silver_clean(df, domain)
                for domain, df in accepted.items()}
for domain, df in silver_clean.items():
    print(domain, ":", df.count(), "validated Silver rows")
    df.printSchema()


## STEP 14 — Save clean, quarantine, and superseded data to Delta Lake

**Goal:** Create 15 named Delta tables (three per source) with persistent lineage.  

**Default safety rule:** Existing tables are **reused, not overwritten**. This is important because LAB 06 will modify/extend the datasets. To reconstruct LAB 05 **before starting LAB 06**, explicitly opt in with `ALLOW_REBUILD_LAB05_TABLES=True` at STEP 01 and re-run top-to-bottom.

**No silent exception handling:** Any failed write or unexpected schema causes the cell to stop. This is a training lab, not a scheduled production job; the five source tables are not committed atomically as a group.

In [ ]:
# STEP 14 — Write 15 Delta datasets without touching LAB 03 SCD2 history.

def store_lab05_snapshot(df, full_table_name):
    """Create a LAB 05 snapshot safely; opt-in rebuild only if explicitly requested."""
    exists = spark.catalog.tableExists(full_table_name)
    if exists and not ALLOW_REBUILD_LAB05_TABLES:
        existing = spark.table(full_table_name)
        assert set(df.columns) == set(existing.columns), (
            f"Existing schema differs for {full_table_name}. Manual review needed."
        )
        assert existing.count() == df.count(), (
            f"Existing rows differ for {full_table_name}. "
            "Review any changes before opting into a rebuild."
        )
        # Compare a stable fingerprint per source row, including the rejection
        # codes where available. This also detects replacement of a row when
        # the table's total row count remains unchanged.
        if "source_record_hash" in df.columns:
            def stable_audit_projection(x):
                fields = ["source_record_hash"]
                if "dq_errors" in x.columns:
                    return x.select(
                        "source_record_hash",
                        F.concat_ws(",", F.array_sort("dq_errors")).alias("audit_dq_codes")
                    )
                return x.select(*fields)
            old_fingerprints = stable_audit_projection(existing)
            new_fingerprints = stable_audit_projection(df)
            assert old_fingerprints.exceptAll(new_fingerprints).limit(1).count() == 0
            assert new_fingerprints.exceptAll(old_fingerprints).limit(1).count() == 0
        print(f"[REUSED] {full_table_name}: {existing.count():,} rows")
        return
    mode = "overwrite" if exists else "errorifexists"
    writer = df.write.format("delta").mode(mode)
    if exists:
        writer = writer.option("overwriteSchema", "true")
    writer.saveAsTable(full_table_name)
    print(f"[{'REBUILT' if exists else 'CREATED'}] {full_table_name}: {df.count():,} rows")

for domain in SOURCE_CONTRACTS:
    store_lab05_snapshot(silver_clean[domain], f"{SILVER}.{domain}_clean")
    store_lab05_snapshot(quarantine[domain], f"{SILVER}.{domain}_quarantine")
    store_lab05_snapshot(superseded[domain], f"{SILVER}.{domain}_superseded")

print("Silver clean/quarantine/superseded tables are available.")


## STEP 15 — Build the persistent Data Quality scorecard

**Goal:** Produce audit-friendly metrics by data source and error code, rather than relying on screenshots. Scorecard outputs support LAB 10 monitoring and an interview demo about source-system accountability.  

Data-quality metrics are **snapshot measurements**, not event-stream metrics or percentages of distinct customers. One row may contribute to more than one error code.

In [ ]:
# STEP 15 — Persist a per-rule quality scorecard for the five source domains.
scorecard_parts = []
for domain in SOURCE_CONTRACTS:
    df_failures = quarantine[domain].select(
        F.lit(domain).alias("domain"),
        F.explode("dq_errors").alias("dq_code")
    )
    scorecard_parts.append(df_failures)

scorecard_events = reduce(lambda a, b: a.unionByName(b), scorecard_parts)
scorecard = (scorecard_events
    .groupBy("domain", "dq_code").count()
    .withColumnRenamed("count", "violation_rows")
    .withColumn("silver_batch_id", F.lit(LAB_RUN_ID)))

store_lab05_snapshot(scorecard, f"{SILVER}.data_quality_rule_metrics_lab05")
store_lab05_snapshot(
    df_reconciliation.withColumn("silver_batch_id", F.lit(LAB_RUN_ID)),
    f"{SILVER}.data_quality_reconciliation_lab05"
)

display(spark.table(f"{SILVER}.data_quality_rule_metrics_lab05")
        .orderBy(F.desc("violation_rows")).limit(25))


## STEP 16 — Validate persisted Delta schemas, uniqueness, and FK integrity

**Goal:** Assert quality properties **after writing** the actual Delta tables. Parent and child checks use Silver only, so a line pointing to a quarantined order is still invalid for the trusted analytical layer.

`UNKNOWN` in Bronze is not a valid child-parent relationship in Silver. This design enables a downstream consumer to join trusted tables without unexpectedly multiplying due to duplicate business keys.

In [ ]:
# STEP 16 — Check uniqueness and relational integrity of published Silver tables.
for domain, (_, expected, key) in SOURCE_CONTRACTS.items():
    fqn = f"{SILVER}.{domain}_clean"
    df = spark.table(fqn)
    assert df.filter(F.col(key).isNull()).count() == 0
    assert df.groupBy(key).count().filter("count > 1").count() == 0
    assert df.filter(F.col("dq_status") != "VALID").count() == 0
    detail = spark.sql(f"DESCRIBE DETAIL {fqn}").select("format").first()[0]
    assert detail.lower() == "delta"
    assert df.count() == next(row["clean_rows"] for row in reconciliation_rows
                              if row["domain"] == domain)
    print(f"PASS: {fqn} has unique keys, valid status and Delta format.")

# Validate the accepted FK graph, not the intentionally imperfect raw graph.
assert (spark.table(f"{SILVER}.orders_clean")
        .join(customer_ref.select("customer_id"), "customer_id", "left_anti").count() == 0)
assert (spark.table(f"{SILVER}.order_items_clean")
        .join(spark.table(f"{SILVER}.orders_clean").select("order_id"),
              "order_id", "left_anti").count() == 0)
assert (spark.table(f"{SILVER}.order_items_clean")
        .join(spark.table(f"{SILVER}.products_clean").select("product_id"),
              "product_id", "left_anti").count() == 0)
assert (spark.table(f"{SILVER}.payments_clean")
        .join(spark.table(f"{SILVER}.orders_clean").select("order_id"),
              "order_id", "left_anti").count() == 0)
assert (spark.table(f"{SILVER}.customer_interactions_clean")
        .join(customer_ref.select("customer_id"), "customer_id", "left_anti").count() == 0)

# Ensure the payment's customer matches the accepted parent order's customer.
matched_payments = (spark.table(f"{SILVER}.payments_clean").alias("p")
    .join(spark.table(f"{SILVER}.orders_clean").alias("o"),
          F.col("p.order_id") == F.col("o.order_id"), "inner"))
assert matched_payments.filter(
    F.col("p.customer_id") != F.col("o.customer_id")
).count() == 0
print("PASS: All Silver foreign-key reference checks completed.")


## STEP 17 — Validate money data types, identity formulas, and source accounting

**Goal:** Check that decimal arithmetic is reliable in published Silver tables. Only *accepted* rows are asserted to follow the local money contracts. We do **not** demand an exact sum match between orders, items, and payments—this is a separate investigation, because failed payment attempts, refunds, rejected child rows, and business lifecycle states complicate the definition of revenue.

In [ ]:
# STEP 17 — Repeat monetary invariants on persisted typed Delta data.
orders_clean = spark.table(f"{SILVER}.orders_clean")
items_clean = spark.table(f"{SILVER}.order_items_clean")
payments_clean = spark.table(f"{SILVER}.payments_clean")

order_diff = F.abs(
    F.col("subtotal_eur") - F.col("discount_eur")
    + F.col("shipping_eur") - F.col("total_amount_eur")
)
assert orders_clean.filter(order_diff > MONEY_TOLERANCE).count() == 0

line_gross_diff = F.abs(F.col("quantity") * F.col("unit_price_eur") - F.col("gross_amount_eur"))
line_net_diff = F.abs(F.col("gross_amount_eur") - F.col("line_discount_eur") - F.col("line_total_eur"))
assert items_clean.filter((line_gross_diff > MONEY_TOLERANCE) |
                          (line_net_diff > MONEY_TOLERANCE)).count() == 0
assert payments_clean.filter(
    ((F.col("payment_type") == "CHARGE") & (F.col("amount_eur") <= 0)) |
    ((F.col("payment_type") == "REFUND") & (F.col("amount_eur") >= 0))
).count() == 0

# Persisted reconciliation must still equal the original 79,000-row snapshot.
for row in reconciliation_rows:
    domain = row["domain"]
    saved = (spark.table(f"{SILVER}.{domain}_clean").count()
             + spark.table(f"{SILVER}.{domain}_quarantine").count()
             + spark.table(f"{SILVER}.{domain}_superseded").count())
    assert saved == row["bronze_rows"], f"Persisted reconciliation failed: {domain}"
print("PASS: Currency math, sign conventions, and 79,000-row accounting.")


## STEP 18 — Order-to-line-items amount reconciliation (diagnostic view)

**Business question:** Does each clean e-commerce order subtotal equal the sum of its **accepted** item totals?  

**Interpretation:** Not necessarily. Some source item lines are invalid or reference rejected products and are deliberately quarantined. A mismatch in this view is an **investigation queue**, not proof of an invalid customer or a reason to silently rewrite the order. Do not count order discounts twice: this model stores discounts at both line and order levels; the order-subtotal source contract is defined by the source manifest.

In [ ]:
# STEP 18 — Create a non-destructive current reconciliation view.
order_reconciliation_view = f"{SILVER}.order_item_reconciliation_lab05"
spark.sql(f"""
CREATE OR REPLACE VIEW {order_reconciliation_view} AS
WITH lines AS (
    SELECT order_id,
           COUNT(*) AS accepted_item_count,
           CAST(SUM(line_total_eur) AS DECIMAL(18,2)) AS accepted_items_total_eur
    FROM {SILVER}.order_items_clean
    GROUP BY order_id
)
SELECT o.order_id, o.customer_id, o.order_status, o.country,
       o.subtotal_eur,
       COALESCE(l.accepted_item_count, 0) AS accepted_item_count,
       l.accepted_items_total_eur,
       CAST(o.subtotal_eur - COALESCE(l.accepted_items_total_eur, 0)
            AS DECIMAL(18,2)) AS subtotal_delta_eur,
       CASE
           WHEN l.order_id IS NULL THEN 'NO_ACCEPTED_ITEMS'
           WHEN ABS(o.subtotal_eur - l.accepted_items_total_eur) > 0.01 THEN 'MISMATCH'
           ELSE 'MATCH'
       END AS reconciliation_status
FROM {SILVER}.orders_clean o
LEFT JOIN lines l ON o.order_id = l.order_id
""")
display(spark.table(order_reconciliation_view)
        .groupBy("reconciliation_status").count().orderBy("reconciliation_status"))
display(spark.table(order_reconciliation_view)
        .filter("reconciliation_status != 'MATCH'")
        .orderBy(F.desc(F.abs(F.col("subtotal_delta_eur")))).limit(20))


## STEP 19 — Payment settlement and refund analysis (diagnostic view)

**Business question:** What portion of each order was **captured and later refunded**?  

One order can generate multiple gateway attempts. Only captured `CHARGE` payments and succeeded `REFUND` payments count toward the illustrative **net settlement** measure. Pending/declined charges are shown as attempts but are not booked as captured revenue.

**Not a revenue-recognition policy:** Accounting revenue depends on fulfilment, cancellations, returns, tax, local standards, and contractual obligations. This query is a transparent financial *exception report* for the interview demo.

In [ ]:
# STEP 19 — Derive settled payment totals by order without double-counting retry attempts.
payment_reconciliation_view = f"{SILVER}.payment_settlement_reconciliation_lab05"
spark.sql(f"""
CREATE OR REPLACE VIEW {payment_reconciliation_view} AS
WITH gateway AS (
    SELECT order_id,
           COUNT(*) AS valid_gateway_events,
           SUM(CASE WHEN payment_type = 'CHARGE' AND payment_status = 'CAPTURED'
                    THEN amount_eur ELSE 0 END) AS captured_charge_eur,
           SUM(CASE WHEN payment_type = 'REFUND' AND payment_status = 'SUCCEEDED'
                    THEN amount_eur ELSE 0 END) AS successful_refund_eur,
           SUM(CASE WHEN payment_type = 'CHARGE' AND payment_status = 'DECLINED'
                    THEN 1 ELSE 0 END) AS declined_attempts
    FROM {SILVER}.payments_clean
    GROUP BY order_id
)
SELECT o.order_id, o.customer_id, o.order_status, o.total_amount_eur,
       COALESCE(g.valid_gateway_events, 0) AS valid_gateway_events,
       CAST(COALESCE(g.captured_charge_eur, 0) AS DECIMAL(18,2)) AS captured_charge_eur,
       CAST(COALESCE(g.successful_refund_eur, 0) AS DECIMAL(18,2)) AS successful_refund_eur,
       CAST(COALESCE(g.captured_charge_eur, 0) + COALESCE(g.successful_refund_eur, 0)
            AS DECIMAL(18,2)) AS net_settled_eur,
       COALESCE(g.declined_attempts, 0) AS declined_attempts
FROM {SILVER}.orders_clean o
LEFT JOIN gateway g ON o.order_id = g.order_id
""")
display(spark.table(payment_reconciliation_view)
        .groupBy("order_status")
        .agg(F.count("*").alias("orders"),
             F.sum("captured_charge_eur").alias("captured_eur"),
             F.sum("successful_refund_eur").alias("refund_eur"),
             F.sum("net_settled_eur").alias("net_settled_eur"))
        .orderBy("order_status"))


## STEP 20 — Separate observable anomalies from blocking quality failures

**Examples of valid but notable observations:**

- `interaction_timestamp` can be much earlier than the source `ingestion_timestamp`. This is **late arrival**, not an invalid interaction.
- `orders.country` can differ from a customer's residential country. It might indicate an international purchase rather than corruption.
- Orders can have quarantined lines or unfinalized payments; report these situations instead of hiding them.

**Definitions must be explicit.** For this lab, a marketing event is "late" when its source ingestion timestamp is **more than 24 hours** after the interaction event timestamp. This threshold is a training assumption.

In [ ]:
# STEP 20A — Explore late-arriving marketing events from accepted Silver records.
interaction_clean = spark.table(f"{SILVER}.customer_interactions_clean")
late_interactions = interaction_clean.withColumn(
    "arrival_delay_hours",
    (F.unix_timestamp("ingestion_timestamp") -
     F.unix_timestamp("interaction_timestamp")) / F.lit(3600.0)
).filter(F.col("arrival_delay_hours") > 24)

print("Accepted late interactions (>24 hours):", late_interactions.count())
display(late_interactions.select(
    "interaction_id", "customer_id", "channel", "interaction_timestamp",
    "ingestion_timestamp", "arrival_delay_hours"
).orderBy(F.desc("arrival_delay_hours")).limit(20))


In [ ]:
# STEP 20B — Country mismatch is an advisory, not a blocking FK error.
order_country_observations = (
    orders_clean.alias("o")
    .join(customer_ref.alias("c"), F.col("o.customer_id") == F.col("c.customer_id"), "inner")
    .select(
        F.col("o.order_id"), F.col("o.customer_id"),
        F.col("o.country").alias("order_country"),
        F.col("c.country").alias("customer_country")
    )
    .filter(F.col("order_country") != F.col("customer_country"))
)
print("Order / customer country differences (advisory):",
      order_country_observations.count())
display(order_country_observations.limit(20))


## STEP 21 — Explore quarantine evidence and domain-specific reject rates

**Goal:** Make your demo explain *why* data is excluded, not just how many records remain. Analysts can inspect individual error codes, normalized values, and the exact raw source payload alongside ingestion hashes. The complete Bronze file remains accessible for audit and replay.

In [ ]:
# STEP 21 — Review the persisted quarantine tables and quality coverage.
for domain in SOURCE_CONTRACTS:
    tbl = f"{SILVER}.{domain}_quarantine"
    q = spark.table(tbl)
    print(f"{domain}: {q.count():,} quarantined record(s)")
    key = SOURCE_CONTRACTS[domain][2]
    display(q.select(key, "dq_errors", "source_file", "source_record_hash")
            .limit(8))

# Compute success ratio: accepted/bronze, separately from local reject rate.
for row in reconciliation_rows:
    accepted_pct = 100 * row["clean_rows"] / row["bronze_rows"]
    print(f"{row['domain']:23s} trusted Silver acceptance: {accepted_pct:.2f}%")


## STEP 22 — Check Unity Catalog objects and Delta history

**Goal:** Demonstrate architectural consistency. `DESCRIBE DETAIL` confirms Delta as the storage format, while transaction history reveals writes/rebuilds. Reconciliation and summary views remain read-only against accepted Silver tables.

**Data governance:** Access to raw payload and quarantine details should be restricted in production because those tables can contain sensitive fields. This synthetic lab uses a public dataset; that does not imply real customer data should be made public.

In [ ]:
# STEP 22 — Inspect Unity Catalog datasets without making destructive changes.
display(spark.sql(f"SHOW TABLES IN {SILVER}"))
display(spark.sql(f"DESCRIBE DETAIL {SILVER}.orders_clean"))
display(spark.sql(f"DESCRIBE HISTORY {SILVER}.orders_clean"))
print("LAB 03 customers_scd2 history has not been modified by this lab.")


## STEP 23 — Final LAB 05 validation checkpoint

A successful execution proves that **all 79,000 Bronze rows were accounted for** across accepted, quarantined, and superseded outcomes; every cleaned table has unique business keys; and all published relational references are valid. It does **not** imply that the original source was accurate or that every financial exception has been resolved.

In [ ]:
# STEP 23 — Final reproducible acceptance criteria.
expected_total = 79000
bronze_total = sum(v[1] for v in SOURCE_CONTRACTS.values())
silver_accounted = sum(
    spark.table(f"{SILVER}.{domain}_clean").count()
    + spark.table(f"{SILVER}.{domain}_quarantine").count()
    + spark.table(f"{SILVER}.{domain}_superseded").count()
    for domain in SOURCE_CONTRACTS
)
assert bronze_total == expected_total
assert silver_accounted == expected_total
assert spark.catalog.tableExists(f"{SILVER}.data_quality_rule_metrics_lab05")
assert spark.catalog.tableExists(f"{SILVER}.data_quality_reconciliation_lab05")
assert spark.catalog.tableExists(f"{SILVER}.order_item_reconciliation_lab05")
assert spark.catalog.tableExists(f"{SILVER}.payment_settlement_reconciliation_lab05")

print("=" * 66)
print("LAB 05 — ADVANCED DATA QUALITY: VALIDATION PASSED")
print("=" * 66)
print(f"Bronze records     : {bronze_total:,}")
print(f"Accounted records  : {silver_accounted:,}")
print("Five clean Delta tables, five quarantine Delta tables,")
print("five superseded Delta tables, two metric Delta tables,")
print("and two reconciliation views are ready for LAB 06/07.")
print("=" * 66)


## STEP 24 — Solutions Architect interview readiness

Use short, business-first explanations; demonstrate actual quarantine rows, a clear FK issue, and one amount-reconciliation report.

**Q1. Why validate foreign keys against Silver instead of raw Bronze?**  
**Suggested answer:** “The Bronze layer preserves what we received, including records that do not meet our contracts. If an order points to a customer that was quarantined during cleansing, that relationship is not trustworthy for analytics. Referencing governed Silver entities prevents invalid upstream records from silently contaminating downstream reports.”

**Q2. What is the difference between data validation and data observability?**  
**Suggested answer:** “Validation defines which rows meet required contracts; observability describes the health and behavior of the data pipelines over time. A late-arriving marketing event may be perfectly valid, but its latency should still be measured and potentially alerted on.”

**Q3. Why not enforce order amount = total payments?**  
**Suggested answer:** “Payments are events, not necessarily one-to-one business facts. There can be declined attempts, pending captures and refunds. I would agree on the business definition—captured payments, net settlement, or accounting revenue—before asserting a reconciliation formula.”

**Q4. How can you prove that your pipeline did not silently discard records?**  
**Suggested answer:** “For each source, I reconcile the source count against the sum of clean, quarantined and superseded records. I also persist rejection codes and ingestion metadata so the data-owner team can investigate and reprocess exceptions.”

**Q5. What changes for production?**  
**Suggested answer:** “I would replace full-snapshot notebook execution with governed incremental pipelines, enforce source data contracts, manage exception remediation, set SLOs and alerts, and use Lakeflow expectations where appropriate. CDC ordering and SCD2 historization belong in a dedicated processing flow.”

**Q6. What should Unity Catalog govern?**  
**Suggested answer:** “Table and Volume access, ownership, discoverability, and lineage. In production I would also consider masking and privilege separation for raw and quarantined personal data, rather than exposing all attributes to business analysts.”

### Suggested 4-minute demo script

1. **Business problem (30 sec):** six systems, inconsistent keys and events, unreliable Customer 360.
2. **Bronze evidence (30 sec):** source values preserved, 79,000 rows and source hashes from GitHub.
3. **Quality engine (90 sec):** standardization, `try_cast`, `dq_errors`, deterministic dedup, FK to `customers_current`.
4. **Outcomes (45 sec):** Silver clean vs quarantine vs superseded, with exact reconciliation.
5. **Value (45 sec):** trusted data and two honest exception views ready for the Gold/AI layer.

**Architecture caveats to state clearly:** LAB 05 is an initial snapshot; LAB 06 adds CDC, MERGE, event ordering and historical change tracking. Deliberate synthetic data issues are useful for interview discussion; they are not incidents from a real customer.

## Official Databricks documentation / next steps

- [Medallion architecture](https://docs.databricks.com/aws/en/lakehouse/medallion)
- [PySpark DataFrames](https://docs.databricks.com/aws/en/pyspark/basics)
- [PySpark SQL built-in functions](https://docs.databricks.com/aws/en/pyspark/reference/functions)
- [TRY_CAST SQL function](https://docs.databricks.com/aws/en/sql/language-manual/functions/try_cast)
- [Unity Catalog](https://docs.databricks.com/aws/en/data-governance/unity-catalog/)
- [Delta table history](https://docs.databricks.com/aws/en/delta/history)
- [Lakeflow Declarative Pipelines expectations](https://docs.databricks.com/aws/en/ldp/expectations)
- [Change data capture and AUTO CDC](https://docs.databricks.com/aws/en/ldp/cdc)

**LAB 06 (next):** Read the three *separate* CDC CSVs in the existing GitHub `data/crm/` folder; de-duplicate by `event_id`, interpret `source_sequence`, handle late/replayed events and tombstones, and apply safe Delta `MERGE` / SCD Type 2 logic without destroying the baseline Silver snapshots.

**LAB 07 (later):** Create Gold `customer_360`, `sales_analytics`, and `customer_segments` from accepted Silver entities, with documented revenue definitions and temporal joins to customers SCD2.